# Análise de Multicolinearidade e Seleção Preliminar de Atributos

**Sprint 1 | Sanidade Vegetal**

A análise combina correlações de Pearson e Spearman, VIF e uma árvore rasa para estimar quais descritores visuais ajudam a separar folhas `HEALTHY` e `RUST`. Correlações e VIF usam os atributos contínuos não-alvo; a árvore usa somente features cromáticas, de textura e nitidez para não transformar tamanho/resolução em atalhos de classe. Todos os cálculos são ajustados no split `train`; validação e teste permanecem fora da seleção. A ABT original não é modificada.

Correlação e VIF identificam redundância linear, não causalidade. A árvore (`max_depth=3`, critério entropia) é uma triagem exploratória e não substitui validação cruzada nem comparação de desempenho do SVM.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'processed' / 'abt_sanidade_vegetal.csv').exists():
            return candidate
    raise FileNotFoundError('ABT não encontrada em data/processed/abt_sanidade_vegetal.csv')

ROOT = find_project_root()
ABT_PATH = ROOT / 'data' / 'processed' / 'abt_sanidade_vegetal.csv'
df = pd.read_csv(ABT_PATH)
FEATURES_VISUAIS = [
    'mean_hue', 'std_saturation', 'exg_index', 'exr_index', 'rg_ratio',
    'indice_clorose_necrose', 'hue_dispersion', 'glcm_contrast',
    'glcm_dissimilarity', 'glcm_homogeneity', 'glcm_energy',
    'indice_rugosidade_pustula', 'laplacian_var'
]
FEATURES_CONTINUAS = ['size_kb', *FEATURES_VISUAIS, 'resolution_mp', 'aspect_ratio']
missing = sorted(set(FEATURES_CONTINUAS + ['class_label', 'split_partition']) - set(df.columns))
if missing:
    raise ValueError(f'Colunas obrigatórias ausentes na ABT: {missing}')

train = df.loc[df['split_partition'].astype(str).str.lower().eq('train')].copy()
if train.empty:
    raise ValueError('Não há registros train; a análise não será ajustada em validação/teste.')
if train[FEATURES_CONTINUAS].isna().any().any():
    raise ValueError('Há valores ausentes nas features contínuas do treino; trate-os antes da análise.')

print(f'ABT: {df.shape[0]:,} linhas x {df.shape[1]} colunas | treino: {len(train):,}')
print('Contínuas na matriz/VIF:', len(FEATURES_CONTINUAS), '| Visuais candidatas para árvore:', len(FEATURES_VISUAIS))
print('Classes HEALTHY/RUST no treino:', train.loc[train['class_label'].isin(['HEALTHY', 'RUST']), 'class_label'].value_counts().to_dict())

## 1. Correlação de Pearson e Spearman

Pearson mede associação linear; Spearman mede associação monotônica por postos e é menos sensível à forma da escala. As duas matrizes são calculadas no treino. Pares com |correlação| >= 0,80 são destacados para revisão, não removidos automaticamente.

In [ ]:
X_continuas = train[FEATURES_CONTINUAS].astype(float)
pearson = X_continuas.corr(method='pearson')
spearman = X_continuas.corr(method='spearman')
print('Matriz de Pearson')
display(pearson.round(3))
print('Matriz de Spearman')
display(spearman.round(3))

def correlation_pairs(matrix, threshold=0.80):
    upper = matrix.where(np.triu(np.ones(matrix.shape), k=1).astype(bool))
    pairs = upper.stack().rename('correlation').reset_index()
    pairs.columns = ['feature_1', 'feature_2', 'correlation']
    return pairs.loc[pairs['correlation'].abs() >= threshold].assign(abs_correlation=lambda data: data['correlation'].abs()).sort_values('abs_correlation', ascending=False)

pearson_pairs = correlation_pairs(pearson)
spearman_pairs = correlation_pairs(spearman)
print('Pares com |Pearson| >= 0,80:')
display(pearson_pairs.round(4).reset_index(drop=True))
print('Pares com |Spearman| >= 0,80:')
display(spearman_pairs.round(4).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(19, 8), constrained_layout=True)
for ax, matrix, title in zip(axes, (pearson, spearman), ('Pearson', 'Spearman')):
    image = ax.imshow(matrix, vmin=-1, vmax=1, cmap='coolwarm', aspect='auto')
    ax.set_xticks(range(len(FEATURES_CONTINUAS)), FEATURES_CONTINUAS, rotation=75, ha='right', fontsize=8)
    ax.set_yticks(range(len(FEATURES_CONTINUAS)), FEATURES_CONTINUAS, fontsize=8)
    ax.set_title(f'Matriz de correlação - {title}')
    for row in range(len(FEATURES_CONTINUAS)):
        for column in range(len(FEATURES_CONTINUAS)):
            ax.text(column, row, f'{matrix.iloc[row, column]:.2f}', ha='center', va='center', fontsize=6)
fig.colorbar(image, ax=axes, shrink=0.8, label='Coeficiente de correlação')
plt.show()

pearson.to_csv(ROOT / 'data' / 'processed' / 'matriz_correlacao_pearson.csv')
spearman.to_csv(ROOT / 'data' / 'processed' / 'matriz_correlacao_spearman.csv')
figures_dir = ROOT / 'docs' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / 'correlacao_pearson_spearman_features.png', dpi=180, bbox_inches='tight')
plt.close(fig)

## 2. Variance Inflation Factor (VIF)

O VIF estima quanto a variância de um coeficiente pode ser inflada pela relação linear de uma feature com as demais. Como referência exploratória, VIF > 5 merece investigação e VIF > 10 indica redundância forte; esses limites não são regras universais. A matriz é centralizada e padronizada apenas para estabilidade numérica, sem alterar o VIF.

In [ ]:
X_vif = (X_continuas - X_continuas.mean()) / X_continuas.std(ddof=0)
if (X_vif.std(ddof=0) == 0).any():
    raise ValueError('VIF indefinido: há feature constante.')
X_vif_const = add_constant(X_vif, has_constant='add')
vif_table = pd.DataFrame({
    'feature': FEATURES_CONTINUAS,
    'vif': [variance_inflation_factor(X_vif_const.to_numpy(), index + 1) for index in range(len(FEATURES_CONTINUAS))]
}).sort_values('vif', ascending=False)
unstable_vif = ~np.isfinite(vif_table['vif']) | vif_table['vif'].ge(1e10)
vif_table['interpretation'] = np.select(
    [unstable_vif, vif_table['vif'] > 10, vif_table['vif'] > 5],
    ['colinearidade perfeita/instabilidade numérica', 'redundância forte (revisar)', 'atenção (investigar)'],
    default='sem alerta pelo limiar adotado'
)
display(vif_table.round(3).reset_index(drop=True))
print(f"Features com VIF > 5 ou não finito: {int(((vif_table['vif'] > 5) | ~np.isfinite(vif_table['vif'])).sum())}")
vif_table.to_csv(ROOT / 'data' / 'processed' / 'vif_features_continuas.csv', index=False)

## 3. Árvore rasa: HEALTHY versus RUST

A árvore é treinada somente no split `train`, com `criterion='entropy'`, profundidade máxima 3 e pesos balanceados por classe. O ranking usa as 13 features visuais (cor, textura e nitidez), não metadados de arquivo/resolução. Árvores não exigem padronização. A importância é baseada na redução de entropia e pode repartir ou concentrar importância entre features correlacionadas.

In [ ]:
train_binary = train.loc[train['class_label'].isin(['HEALTHY', 'RUST'])].copy()
if train_binary['class_label'].nunique() != 2:
    raise ValueError('O treino precisa conter as classes HEALTHY e RUST.')
tree = DecisionTreeClassifier(criterion='entropy', max_depth=3, class_weight='balanced', random_state=42)
tree.fit(train_binary[FEATURES_VISUAIS], train_binary['class_label'])
ranking = pd.DataFrame({
    'feature': FEATURES_VISUAIS,
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)
ranking.insert(0, 'rank', np.arange(1, len(ranking) + 1))
display(ranking.round(5))

plot_ranking = ranking.sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_ranking['feature'], plot_ranking['importance'], color='#287D8E')
ax.set_title('Importância de atributos | árvore rasa HEALTHY vs RUST')
ax.set_xlabel('Importância (redução de entropia normalizada)')
ax.set_ylabel('Feature')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()
fig.savefig(figures_dir / 'importancia_features_arvore_healthy_rust.png', dpi=180, bbox_inches='tight')
plt.close(fig)
ranking.to_csv(ROOT / 'data' / 'processed' / 'ranking_features_healthy_rust.csv', index=False)
print(f'Árvore: profundidade efetiva {tree.get_depth()}, folhas {tree.get_n_leaves()}, amostras {len(train_binary):,}.')

## 4. Síntese para o Relato da Solução

Use os pares de correlação e o VIF para apontar grupos redundantes, e o ranking da árvore para priorizar candidatos a ablação. Correlação alta entre duas features não prova que uma delas seja inútil; compare remoções em validação cruzada dentro do pipeline anti-leakage. Para SVM RBF, padronize as features no pipeline e avalie subconjuntos por CV. A árvore fornece uma triagem supervisionada simples, não uma estimativa final de desempenho. Os CSVs e figuras abaixo são derivados; os dados fonte permanecem intactos.

In [ ]:
high_corr_pearson = pearson_pairs.head(5)
high_corr_spearman = spearman_pairs.head(5)
high_vif = vif_table.loc[(vif_table['vif'] > 5) | ~np.isfinite(vif_table['vif'])]
top_features = ranking.loc[ranking['importance'] > 0, 'feature'].head(5).tolist()
print(f'Treino: {len(train):,} amostras; atributos contínuos analisados: {len(FEATURES_CONTINUAS)}.')
print(f'Pares Pearson com |r| >= 0,80: {len(pearson_pairs)}; pares Spearman com |rho| >= 0,80: {len(spearman_pairs)}.')
print('Pares redundantes mais fortes (Pearson):')
display(high_corr_pearson[['feature_1', 'feature_2', 'correlation']].round(4).reset_index(drop=True))
print('Pares redundantes mais fortes (Spearman):')
display(high_corr_spearman[['feature_1', 'feature_2', 'correlation']].round(4).reset_index(drop=True))
print(f'Features com VIF > 5/não finito: {len(high_vif)}; VIF >= 1e10 indica singularidade/instabilidade numérica.')
display(vif_table[['feature', 'vif', 'interpretation']].head(8).round(3).reset_index(drop=True))
print('Ranking positivo da árvore HEALTHY/RUST:')
display(ranking.loc[ranking['importance'] > 0].head(5).round(5).reset_index(drop=True))
print('Recomendação: revisar redundâncias; priorizar ablação por validação cruzada no treino e manter o teste intocado.')
print('Arquivos: matriz_correlacao_pearson.csv, matriz_correlacao_spearman.csv, vif_features_continuas.csv, ranking_features_healthy_rust.csv')
print('Figuras: docs/figures/correlacao_pearson_spearman_features.png e docs/figures/importancia_features_arvore_healthy_rust.png')